# Extracting structured data from a resume

This example shows how the same `parse()` pipeline can be used on a resume:

1. **Raw parse** - turn the document into clean markdown (static parser, no API key).
2. **Schema-guided extraction** - ask a model to fill a schema, so the output is typed
   JSON you can feed into an application.

The sample resume in `inputs/resume_sample.pdf` is fully synthetic.


In [ ]:
from lexoid.api import parse, parse_with_schema

RESUME_PATH = "inputs/resume_sample.pdf"


## 1. Raw parse

Static parsing works offline and needs no credentials.


In [1]:
raw = parse(RESUME_PATH, parser_type="STATIC_PARSE")["raw"]
print(raw[:800])


# **Alex Morgan**

### Senior Backend Engineer

 <alex.morgan@example.com> | +1 (555) 010-2030 | Austin, TX

### **Summary**

 Backend engineer with 8 years of experience designing distributed services and data pipelines. Focused on

 reliability, observability, and cost efficiency in cloud-native systems. Comfortable owning a service end to end, from

 schema design to on-call.

### **Skills**

 l **Languages:** Python, Go, SQL, TypeScript

 l **Infrastructure:** Kubernetes, Docker, Terraform, AWS (EKS, S3, Lambda, RDS)

 l **Data:** PostgreSQL, Redis, Kafka, Snowflake, dbt

### **Experience**

 **Northwind Data** - Senior Backend Engineer

 Mar 2021 - Present | Austin, TX

 l Led the migration of the ingestion platform from a monolith to event-driven services on Kafka, cutting p95

&nbsp


## 2. Schema-guided extraction

Define the fields you care about and let a model fill them. The example uses
`gpt-4o-mini`; pass any other supported model/provider to `parse_with_schema`
(for example `api="local"` with an Ollama model, or another OpenAI-compatible
endpoint via `OPENAI_BASE_URL`).


In [ ]:
from typing import List, Optional

from pydantic import BaseModel, Field


class Experience(BaseModel):
    company: str = Field(description="Employer name")
    title: str = Field(description="Job title")
    start_date: str = Field(description="Start date as written in the resume")
    end_date: Optional[str] = Field(None, description="End date, or 'Present'")
    highlights: List[str] = Field(default_factory=list, description="Key achievements")


class Education(BaseModel):
    degree: str
    institution: str
    graduation_year: Optional[int] = None


class Resume(BaseModel):
    name: str = Field(description="Candidate full name")
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    summary: Optional[str] = Field(None, description="Short professional summary")
    skills: List[str] = Field(default_factory=list)
    experience: List[Experience] = Field(default_factory=list)
    education: List[Education] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)


In [ ]:
import json

resume = parse_with_schema(path=RESUME_PATH, schema=Resume, model="gpt-4o-mini")
print(json.dumps(resume, indent=2, default=str))


## Notes

- The schema path renders the document to images and sends them to a vision-capable model,
  so it expects a PDF/image input plus a multimodal provider. `parse()` itself also handles
  DOCX, XLSX, PPTX and URLs.
- If a field comes back empty, make its `description=` more explicit before reaching for a
  bigger model.
- The extraction cell is left unexecuted here because it needs a model provider; the raw
  parse cell above runs fully offline.
